In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
from collections import defaultdict
from collections import Counter


# ---------------------------------------------------------------------
# Repository root
# ---------------------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# ---------------------------------------------------------------------
# Base paths
# ---------------------------------------------------------------------
BASE_DIR = ROOT / "data"
PREPROCESSED_DIR = BASE_DIR / "processed" / "preprocessed_features"
MOTIF_DIR = BASE_DIR / "raw" / "Beethoven_motif" / "motif_notes"
OUTPUT_DIR = BASE_DIR / "processed" / "transformations_outputs"
MATRIX_DIR = OUTPUT_DIR / "transformation_matrices"
FEATURE_DIR = OUTPUT_DIR / "segment_level_features"

MATRIX_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def match_motif_instances(df_motifs_only, motif_folder):
    # Ensure motif_folder is a Path object
    motif_folder = Path(motif_folder)

    # Final output: { motif_type: [DataFrame, DataFrame, ...] }
    motif_groups = {}

    # Iterate through motif instance files
    for file in motif_folder.iterdir():
        if file.suffix != ".csv":
            continue

        motif_file_name = file.stem       # e.g. A-1
        motif_file_key = motif_file_name.lower()    # e.g. a-1
        motif_type = motif_file_key.split("-")[0]   # e.g. "a"

        # Subset the data to only that motif type
        df_same_type = df_motifs_only[df_motifs_only["motif"] == motif_type]

        if df_same_type.empty:
            continue  # Skip if no such motif type in the preprocessed data

        # Load current motif instance CSV
        motif_df = pd.read_csv(file, header=None)
        motif_df.columns = ["onset", "midi_number", "morphetic_number", "duration", "staff_number", "measure"]

        matched_rows = []
        unmatched_rows = []

        # --- First pass: strict match on onset ±0.05, exact pitch and staff
        for _, row in motif_df.iterrows():
            onset, pitch, staff = row["onset"], row["midi_number"], row["staff_number"]
            match = df_same_type[
                (df_same_type["pitch"] == pitch) &
                (df_same_type["clef"] == staff) &
                (df_same_type["onset"].sub(onset).abs() <= 0.05)
            ]
            if not match.empty:
                matched_rows.extend(match.index)
            else:
                unmatched_rows.append((onset, pitch, staff))

        # --- Second pass: allow pitch ±1
        for onset, pitch, staff in unmatched_rows.copy():
            match = df_same_type[
                (df_same_type["clef"] == staff) &
                (df_same_type["onset"].sub(onset).abs() <= 0.05) &
                (df_same_type["pitch"].sub(pitch).abs() <= 1)
            ]
            if not match.empty:
                matched_rows.extend(match.index)
                unmatched_rows.remove((onset, pitch, staff))

        # --- Third pass: larger onset window ±0.5, exact pitch
        for onset, pitch, staff in unmatched_rows:
            match = df_same_type[
                (df_same_type["clef"] == staff) &
                (df_same_type["onset"].sub(onset).abs() <= 0.5) &
                (df_same_type["pitch"] == pitch)
            ]
            if not match.empty:
                matched_rows.extend(match.index)

        # --- Fourth pass: larger onset window ±2, exact pitch
        for onset, pitch, staff in unmatched_rows:
            match = df_same_type[
                (df_same_type["clef"] == staff) &
                (df_same_type["onset"].sub(onset).abs() <= 2) &
                (df_same_type["pitch"] == pitch)
            ]
            if not match.empty:
                matched_rows.extend(match.index)

        matched_df = df_same_type.loc[sorted(set(matched_rows))].copy()

        # Append matched instance to its motif type group
        if motif_type not in motif_groups:
            motif_groups[motif_type] = []

        motif_groups[motif_type].append(matched_df)

    return motif_groups

In [ ]:
def sort_motif_instances(motif_groups):
    # Sort motif instances within each motif type by first onset
    for motif_type in motif_groups:
        motif_groups[motif_type].sort(key=lambda df: df["onset"].min())

    # Ensure all motif instances are sorted by onset
    for motif_class, instance_list in motif_groups.items():
        for i in range(len(instance_list)):
            instance_list[i] = instance_list[i].sort_values(by='onset').reset_index(drop=True)

    return motif_groups


In [ ]:
def detect_sentence_boundaries(df, min_measures=8, start_idx=0):
    sentence_bounds = []
    notes_df = df[df["type"] == "Note"].copy().reset_index(drop=True)
    note_intervals = notes_df[["onset", "offset"]].to_numpy()
    rest_intervals = df[df["type"] == "Rest"][["onset", "offset"]].to_numpy()

    # --- Silence detection ---
    def is_silent(start, end):
        return not np.any((note_intervals[:, 0] < end) & (note_intervals[:, 1] > start))

    silence_onsets = {
        start for start, end in rest_intervals if is_silent(start, end)
    }

    # --- Repeated pitch (1+ beat gap) ---
    repeated_pitch_onsets = set(
        notes_df.loc[
            (notes_df["pitch"] == notes_df["pitch"].shift(-1)) &
            ((notes_df["onset"].shift(-1) - notes_df["onset"]) >= 1),
            "onset"
        ]
    )

    # --- Cadence logic ---
    cadence_onsets = set()
    candidates = notes_df[[
        "onset", "scale_degree_semitone_offset", "chord_inversion",
        "is_major", "is_minor", "is_augmented_sixth", "is_neapolitan"
    ]].copy()
    candidates["next_deg"] = candidates["scale_degree_semitone_offset"].shift(-1)
    candidates["next_onset"] = candidates["onset"].shift(-1)

    for _, row in candidates.iterrows():
        cur, nxt = row["scale_degree_semitone_offset"], row["next_deg"]
        if pd.isna(cur) or pd.isna(nxt) or cur == nxt or cur == -1 or nxt == -1:
            continue

        if cur == 7 and nxt == 0 and row["chord_inversion"] == 0:  # Authentic
            cadence_onsets.add(row["next_onset"])
        elif nxt == 7 and cur != 7:  # Half
            cadence_onsets.add(row["next_onset"])
        elif cur == 5 and nxt == 0:  # Plagal
            cadence_onsets.add(row["next_onset"])
        elif cur == 7 and nxt in [8, 9]:  # Deceptive
            cadence_onsets.add(row["next_onset"])
        elif row["is_augmented_sixth"] and nxt == 7:
            cadence_onsets.add(row["next_onset"])
        elif row["is_neapolitan"] and nxt in [0, 7]:
            cadence_onsets.add(row["next_onset"])

    # --- Prioritize boundaries ---
    ordered_boundaries = sorted(
        [{"onset": o, "type": "silence"} for o in silence_onsets] +
        [{"onset": o, "type": "repeated"} for o in repeated_pitch_onsets] +
        [{"onset": o, "type": "cadence"} for o in cadence_onsets],
        key=lambda x: x["onset"]
    )

    # --- Motif safety mask: drop boundaries that fall inside any motif instance ---
    motif_intervals = df[~df["motif"].isna()][["onset", "offset"]].to_numpy()

    def is_inside_any_motif(onset):
        return np.any((motif_intervals[:, 0] <= onset) & (onset < motif_intervals[:, 1]))

    filtered_boundaries = [
        b for b in (
            [{"onset": o, "type": "silence"} for o in silence_onsets] +
            [{"onset": o, "type": "repeated"} for o in repeated_pitch_onsets] +
            [{"onset": o, "type": "cadence"} for o in cadence_onsets]
        )
        if not is_inside_any_motif(b["onset"])
    ]

    ordered_boundaries = sorted(filtered_boundaries, key=lambda x: x["onset"])

    # --- Main segmentation loop (unchanged) ---
    while start_idx < len(notes_df):
        start_onset = notes_df.loc[start_idx, "onset"]
        start_meas = notes_df.loc[start_idx, "start_meas"]
        candidate = None

        for i, b in enumerate(ordered_boundaries):
            if b["onset"] <= start_onset:
                continue
            idx_b = notes_df[notes_df["onset"] >= b["onset"]].index
            if idx_b.empty:
                continue
            boundary_meas = notes_df.loc[idx_b[0], "start_meas"]
            if boundary_meas < start_meas + min_measures:
                continue

            # Check if a better (silence or repeated) boundary exists within 2 later onsets
            if b["type"] == "cadence":
                lookahead = ordered_boundaries[i + 1:i + 3]
                if any(x["type"] in {"silence", "repeated"} for x in lookahead):
                    continue

            candidate = b["onset"]
            break

        if candidate is not None:
            sentence_bounds.append((start_onset, candidate))
            next_idx = notes_df[notes_df["onset"] > candidate].index
            if not next_idx.empty:
                start_idx = next_idx[0]
            else:
                break
        else:
            sentence_bounds.append((start_onset, notes_df["offset"].iloc[-1]))
            break
    
    return sentence_bounds


In [ ]:
def build_segment_dfs(df, sentence_bounds):
    segment_dfs = {}

    for i, (start, end) in enumerate(sentence_bounds):
        segment = df[(df["onset"] >= start) & (df["onset"] < end)].copy()
        if not segment.empty:
            segment_dfs[i] = segment
    return segment_dfs


In [ ]:
def assign_motifs_to_segments(motif_groups, sentence_bounds):
    # Step 1: Map segment index to onset range
    segment_index_bounds = {
        idx: (start, end)
        for idx, (start, end) in enumerate(sentence_bounds)
    }

    # Step 2: Assign motif instances to segments
    final_segment_motif_dict = defaultdict(lambda: defaultdict(list))

    for motif_class, instance_list in motif_groups.items():
        for instance_df in instance_list:
            onset = instance_df['onset'].min()
            for seg_idx, (start, end) in segment_index_bounds.items():
                if start <= onset < end:
                    final_segment_motif_dict[seg_idx][motif_class].append(instance_df)
                    break

    # Optional: convert defaultdicts to regular dicts
    final_segment_motif_dict = {
        k: dict(v) for k, v in final_segment_motif_dict.items()
    }
    return final_segment_motif_dict

In [ ]:
def compute_motif_level_features(final_segment_motif_dict):
    motif_feature_rows = []

    for segment_id, motif_dict in final_segment_motif_dict.items():
        for motif_class, motif_instances in motif_dict.items():
            for motif_df in motif_instances:
                feats = {}

                if motif_df.empty:
                    continue

                motif_df = motif_df.copy()
                motif_df_unique = motif_df.drop_duplicates()
                seg_duration = motif_df_unique['scaled_duration'].sum()
                seg_duration = seg_duration if seg_duration > 0 else 1.0

                onset = motif_df["onset"].min()
                feats["segment_id"] = segment_id
                feats["motif_class"] = motif_class
                feats["onset"] = onset

                # --- Harmonic Features ---
                harmony = motif_df_unique[[
                    'scale_degree_semitone_offset', 'secondary_scale_degree_semitone_offset',
                    'is_major', 'is_minor', 'is_diminished', 'is_augmented',
                    'is_seventh', 'chord_inversion',
                    'is_augmented_sixth', 'is_neapolitan',
                    'harmonic_complexity'
                ]].drop_duplicates()

                feats['spread_harmonic_complexity'] = harmony['harmonic_complexity'].max() - harmony['harmonic_complexity'].min()
                total_chords = len(harmony)
                n_secondary = (harmony['secondary_scale_degree_semitone_offset'] != -1).sum()
                feats['secondary_chord_proportion'] = n_secondary / total_chords if total_chords > 0 else 0

                # --- Key Change Count ---
                key_df = motif_df_unique[['localkey_fifths', 'localkey_mode']].drop_duplicates()
                feats['key_change_count'] = (
                    (key_df['localkey_fifths'].shift() != key_df['localkey_fifths']) |
                    (key_df['localkey_mode'].shift() != key_df['localkey_mode'])
                ).sum() / seg_duration

                # --- Pitch Features ---
                pitch_vals = motif_df_unique['pitch'].replace(-1, np.nan).dropna()
                feats['pitch_spread'] = pitch_vals.max() - pitch_vals.min()
                feats['motif_pitch_register'] = motif_df_unique['relative_pitch_scale'].dropna().median()

                # --- Rhythmic: IOI ---
                ioi_vals = motif_df_unique['normalized_ioi'].dropna()
                # feats['mean_ioi'] = ioi_vals.mean()
                if len(ioi_vals) < 2:
                    feats['std_ioi'] = 0.0
                else:
                    feats['std_ioi'] = float(ioi_vals.std())

                # --- Rhythmic: Silence Proportion ---
                motif_df_sorted = motif_df_unique.sort_values("onset")
                onset_diff = np.diff(motif_df_sorted["onset"])
                silence_durations = onset_diff[onset_diff > 0.01]
                feats['silence_proportion'] = silence_durations.sum() / seg_duration if seg_duration > 0 else 0

                # --- Rhythmic: Metrical Stress Rate ---
                def compute_metrical_strength(row):
                    ts = (row['timesig_numerator'], row['timesig_denominator'])
                    weights_map = {
                        (2, 2): [1.0, 0.5], (3, 4): [1.0, 0.5, 0.5],
                        (3, 8): [1.0, 0.5, 0.5], (4, 4): [1.0, 0.5, 0.75, 0.25],
                        (6, 4): [1.0, 0.5, 0.75, 0.25, 0.5, 0.25],
                        (6, 8): [1.0, 0.5, 0.75, 0.25, 0.5, 0.25],
                        (12, 8): [1.0, 0.5, 0.75, 0.25, 0.5, 0.25, 0.75, 0.25, 0.5, 0.25, 0.5, 0.25],
                        (13, 8): [1.0, 0.5, 0.5, 0.25] + [0.25] * 9
                    }
                    beat = row['beat']
                    if not float(beat).is_integer():
                        return np.nan
                    idx = int(beat)
                    return weights_map.get(ts, [0.5] * 16)[idx] if idx < len(weights_map.get(ts, [])) else 0.25

                # Apply to all rows
                metrical_strengths = motif_df_unique.apply(compute_metrical_strength, axis=1)

                # Fallback if all values were NaN
                if metrical_strengths.dropna().empty:
                    feats['metrical_stress_rate'] = 0.25  # fallback for all non-integer beats
                else:
                    feats['metrical_stress_rate'] = metrical_strengths.dropna().mean()

                # --- Expressive: Density & Accentuation ---
                expressive_cols = ['is_slur', 'is_cresc', 'is_dim', 'is_accent']
                feats['expressive_density'] = motif_df_unique[expressive_cols].mean(axis=1).mean()
                dyn_vals = motif_df_unique['ordinal_dynamic'].replace(-1, np.nan)
                if len(dyn_vals) < 2:
                    feats['accentuation_std'] = 0.0
                else:
                    feats['accentuation_std'] = float(dyn_vals.std())

                motif_feature_rows.append(feats)

    df = pd.DataFrame(motif_feature_rows)
    df.sort_values("onset", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


In [ ]:
def compute_anchor_motifs(final_segment_motif_dict):
    segment_anchor_motifs = defaultdict(dict)
    anchor_metadata = {}
    previous_anchor = None

    for segment_idx in sorted(final_segment_motif_dict.keys()):
        motif_classes = final_segment_motif_dict[segment_idx]

        for motif_class, instance_list in motif_classes.items():
            if len(instance_list) > 1:
                anchor = instance_list[0]
                segment_anchor_motifs[segment_idx][motif_class] = anchor
                anchor_metadata[(segment_idx, motif_class)] = {"source": "first_in_segment"}
                previous_anchor = anchor
            elif previous_anchor is not None:
                segment_anchor_motifs[segment_idx][motif_class] = previous_anchor
                anchor_metadata[(segment_idx, motif_class)] = {"source": "previous_anchor"}
            else:
                anchor = instance_list[0]
                segment_anchor_motifs[segment_idx][motif_class] = anchor
                anchor_metadata[(segment_idx, motif_class)] = {"source": "self_assigned_first_segment"}
                previous_anchor = anchor

    return segment_anchor_motifs, anchor_metadata


In [ ]:
def align_motifs(anchor_df, motif_df, beat_tol=1e-3, dur_tol=1e-3, alpha=1.0, beta=1.0):
    """
    Align notes between anchor_df and motif_df using multiple strategies.
    Case 5 now uses dynamic programming to preserve the same contour-aware
    cost definition as the original combinatorial method.

    Parameters:
        alpha: weight for pitch contour distance
        beta: weight for beat contour distance
    """
    anchor_df = anchor_df.sort_values("onset").reset_index(drop=True)
    motif_df = motif_df.sort_values("onset").reset_index(drop=True)

    n, m = len(anchor_df), len(motif_df)
    use_anchor_as_ref = n <= m
    shorter_df = anchor_df if use_anchor_as_ref else motif_df
    longer_df = motif_df if use_anchor_as_ref else anchor_df

    shorter_len = len(shorter_df)
    longer_len = len(longer_df)

    def pack(i, j):
        return (i, j) if use_anchor_as_ref else (j, i)

    # ---------------------
    # Case 1: Equal length → direct pairing
    # ---------------------
    if shorter_len == longer_len:
        return [pack(i, i) for i in range(shorter_len)]

    # ---------------------
    # Case 2: Exact pitch subsequence match
    # ---------------------
    short_pitch_seq = shorter_df['pitch'].tolist()
    long_pitch_seq = longer_df['pitch'].tolist()
    for start_idx in range(longer_len - shorter_len + 1):
        window = long_pitch_seq[start_idx:start_idx + shorter_len]
        if window == short_pitch_seq:
            return [pack(i, start_idx + i) for i in range(shorter_len)]

    # ---------------------
    # Case 3: Match by scaled_beat
    # ---------------------
    alignment = []
    used_j = -1
    for i, beat in enumerate(shorter_df['scaled_beat']):
        candidates = longer_df[
            (np.abs(longer_df['scaled_beat'] - beat) < beat_tol) &
            (longer_df.index > used_j)
        ]
        if candidates.empty:
            break
        j = candidates.index[0]
        alignment.append(pack(i, j))
        used_j = j
    if len(alignment) == shorter_len:
        return alignment

    # ---------------------
    # Case 4: Match by scaled_duration
    # ---------------------
    alignment = []
    used_j = -1
    for i, dur in enumerate(shorter_df['scaled_duration']):
        candidates = longer_df[
            (np.abs(longer_df['scaled_duration'] - dur) < dur_tol) &
            (longer_df.index > used_j)
        ]
        if candidates.empty:
            break
        j = candidates.index[0]
        alignment.append(pack(i, j))
        used_j = j
    if len(alignment) == shorter_len:
        return alignment

    # ---------------------
    # Case 5: Contour-aware alignment using DP (pitch + beat)
    # ---------------------
    short_pitches = shorter_df['pitch'].values
    short_beats = shorter_df['scaled_beat'].values
    long_pitches = longer_df['pitch'].values
    long_beats = longer_df['scaled_beat'].values

    n, m = shorter_len, longer_len
    dp = np.full((n + 1, m + 1), np.inf)
    back = np.full((n + 1, m + 1), -1, dtype=int)
    dp[0, :] = 0.0

    # DP: cost = contour difference (pitch and beat)
    for i in range(1, n + 1):
        for j in range(i, m + 1):  # maintain order
            if i == 1:
                # First note has no contour; cost is 0
                cost = 0
            else:
                # Calculate contour differences based on chosen j and previous alignment
                prev_j = back[i - 1, j - 1]
                if prev_j == -1:
                    continue
                pitch_contour_short = short_pitches[i - 1] - short_pitches[i - 2]
                beat_contour_short = short_beats[i - 1] - short_beats[i - 2]
                pitch_contour_long = long_pitches[j - 1] - long_pitches[prev_j]
                beat_contour_long = long_beats[j - 1] - long_beats[prev_j]
                cost = alpha * abs(pitch_contour_short - pitch_contour_long) + \
                       beta * abs(beat_contour_short - beat_contour_long)

            # Update DP table if this alignment improves cost
            if dp[i - 1, j - 1] + cost < dp[i, j]:
                dp[i, j] = dp[i - 1, j - 1] + cost
                back[i, j] = j - 1

    # Recover best alignment
    if np.isinf(dp[n]).all():
        return []

    j = np.argmin(dp[n])
    best_alignment = []
    for i in range(n, 0, -1):
        best_alignment.append(pack(i - 1, back[i, j]))
        j = back[i, j]

    return best_alignment[::-1]


In [ ]:
def evaluate_identity_transformation(anchor_df, motif_df, alignment, tol=1e-6):
    """
    Identity transformation under alignment:
      - Reject if the full onset vectors are identical (same instance)
      - Compare pitch + scaled_duration only for aligned notes
      - Vectorized, no loops, no operations field

    Returns:
        {
            "preserved": True/False,
            "reason": <str>
        }
    """

    # 1. Reject if motif is literally the same occurrence (same onset vector)
    if (
        anchor_df["onset"].shape == motif_df["onset"].shape and
        np.allclose(anchor_df["onset"].to_numpy(), motif_df["onset"].to_numpy())
    ):
        return {
            "preserved": False,
            "reason": "Motif has identical onset vector to anchor (same instance)"
        }

    # 2. Need alignment to compare notes
    if len(alignment) == 0:
        return {
            "preserved": False,
            "reason": "No aligned notes"
        }

    # 3. Vectorize aligned-index extraction
    anchor_idx = np.array([a for a, _ in alignment], dtype=int)
    motif_idx  = np.array([m for _, m in alignment], dtype=int)

    a_pitch = anchor_df.loc[anchor_idx, "pitch"].to_numpy()
    m_pitch = motif_df.loc[motif_idx, "pitch"].to_numpy()

    a_dur   = anchor_df.loc[anchor_idx, "scaled_duration"].to_numpy()
    m_dur   = motif_df.loc[motif_idx, "scaled_duration"].to_numpy()

    # 4. Vectorized equality check
    equal_pitch = np.abs(a_pitch - m_pitch) <= tol
    equal_dur   = np.abs(a_dur   - m_dur)   <= tol

    equal_vec = equal_pitch & equal_dur

    # 5. Decision
    if np.all(equal_vec):
        return {
            "preserved": True,
            "reason": "Aligned pitch + duration identical"
        }

    return {
        "preserved": False,
        "reason": "Aligned pitch/duration differ"
    }


In [ ]:
def evaluate_contour_preserving(anchor_df, motif_df, alignment):
    """
    Evaluate Contour-Preserving transformation, refined.

    A transformation is considered contour-preserving if:
    - The sign of pitch intervals (Δpitch) is preserved.
    - The intervals differ between anchor and motif.
    - The interval difference is significant (non-smooth interval pattern).

    Parameters:
        anchor_df : DataFrame of the anchor motif.
        motif_df  : DataFrame of the transformed motif.
        alignment : List of (anchor_idx, motif_idx) tuples.

    Returns:
        dict with:
            - preserved : True/False/None
            - reason    : explanation
            - operations: list of interval comparison dicts
    """
    if len(alignment) < 2:
        return {"preserved": None, "operations": None, "reason": "Too few aligned notes"}

    anchor_pitches = anchor_df["pitch"].values
    motif_pitches = motif_df["pitch"].values

    alignment = sorted(alignment, key=lambda x: x[0])
    anchor_idxs, motif_idxs = zip(*alignment)

    anchor_aligned = anchor_pitches[list(anchor_idxs)]
    motif_aligned = motif_pitches[list(motif_idxs)]

    anchor_intervals = np.diff(anchor_aligned)
    motif_intervals = np.diff(motif_aligned)

    anchor_contour = np.sign(anchor_intervals)
    motif_contour = np.sign(motif_intervals)
    contour_match = np.array_equal(anchor_contour, motif_contour)

    interval_diffs = np.abs(anchor_intervals - motif_intervals)
    non_smooth_interval_pattern = np.mean(interval_diffs) > 0.5

    if contour_match and non_smooth_interval_pattern:
        preserved = True
        reason = "Contour preserved with non-smooth interval pattern"
    elif contour_match and not non_smooth_interval_pattern:
        preserved = False
        reason = "Contour preserved but intervals are too similar (smooth shape)"
    else:
        preserved = False
        reason = "Contour direction changed"

    operations = [
        {"anchor_interval": float(a), "motif_interval": float(m)}
        for a, m in zip(anchor_intervals, motif_intervals)
    ]

    return {
        "preserved": preserved,
        "operations": operations,
        "reason": reason
    }


In [ ]:
def evaluate_salient_leap_preserving(anchor_df, motif_df, alignment, threshold=2, epsilon=0):
    """
    Evaluate Salient Leap-Preserving Transformation based on second derivative of pitch.

    Parameters:
    - anchor_df, motif_df: DataFrames containing aligned motifs (must have 'pitch').
    - alignment: list of (anchor_idx, motif_idx) tuples, in order.
    - threshold: salience threshold on |Δ² pitch| (e.g., 2)
    - epsilon: allowed mismatch in count of salient leaps

    Returns:
    A dictionary with:
    - preserved: bool (True only if leap count matches and anchor has at least one)
    - num_anchor_salient: int
    - num_motif_salient: int
    - reason: explanation
    """

    if len(alignment) < 3:
        return {
            "preserved": None,
            "num_anchor_salient": 0,
            "num_motif_salient": 0,
            "reason": "Too few aligned notes to compute second derivative"
        }

    # Sort alignment and extract pitch sequences
    alignment = sorted(alignment, key=lambda x: x[0])
    anchor_idxs, motif_idxs = zip(*alignment)
    anchor_pitches = anchor_df["pitch"].values[list(anchor_idxs)]
    motif_pitches = motif_df["pitch"].values[list(motif_idxs)]

    # Compute second derivatives (Δ² pitch)
    anchor_d2 = np.diff(np.diff(anchor_pitches))
    motif_d2 = np.diff(np.diff(motif_pitches))

    # Count salient leaps
    num_anchor_salient = np.sum(np.abs(anchor_d2) > threshold)
    num_motif_salient = np.sum(np.abs(motif_d2) > threshold)

    if np.array_equal(anchor_pitches, motif_pitches):
        return {
            "preserved": False,
            "num_anchor_salient": int(np.sum(np.abs(np.diff(np.diff(anchor_pitches))) > threshold)),
            "num_motif_salient": int(np.sum(np.abs(np.diff(np.diff(motif_pitches))) > threshold)),
            "reason": "Motif is identical to anchor"
        }

    if num_anchor_salient == 0:
        return {
            "preserved": False,
            "num_anchor_salient": int(num_anchor_salient),
            "num_motif_salient": int(num_motif_salient),
            "reason": "Anchor has no salient leaps to preserve"
        }

    same_num = abs(num_anchor_salient - num_motif_salient) <= epsilon

    return {
        "preserved": same_num,
        "num_anchor_salient": int(num_anchor_salient),
        "num_motif_salient": int(num_motif_salient),
        "reason": (
            "Salient leap counts match" if same_num else
            f"Anchor has {num_anchor_salient}, motif has {num_motif_salient} salient leaps"
        )
    }


In [ ]:
def evaluate_rhythm_preserving(anchor_df, motif_df, alignment, tol=1e-4):
    """
    Evaluate Rhythm-Preserving transformation based on interval difference homogeneity.

    Uses:
    - Median of interval differences
    - Mean absolute deviation (MAD) from the median
    - Preservation score = 1 - (MAD / effective denominator)

    Returns:
    - preserved: boolean indicating if preserved_score >= 0.75
    - preserved_score: float in [0, 1]
    - anchor_intervals: list of anchor beat intervals
    - motif_intervals: list of motif beat intervals
    - scaling_diffs: motif - anchor differences
    - reason: explanation string
    """
    if len(alignment) < 2:
        return {
            "preserved": None,
            "preserved_score": None,
            "anchor_intervals": [],
            "motif_intervals": [],
            "scaling_diffs": [],
            "reason": "Too few aligned notes"
        }

    anchor_df = anchor_df.sort_values("onset").reset_index(drop=True)
    motif_df = motif_df.sort_values("onset").reset_index(drop=True)
    alignment = sorted(alignment, key=lambda x: x[0])
    anchor_idxs, motif_idxs = zip(*alignment)

    anchor_beats = anchor_df["scaled_beat"].values[list(anchor_idxs)]
    motif_beats = motif_df["scaled_beat"].values[list(motif_idxs)]

    anchor_intervals = np.diff(anchor_beats)
    motif_intervals = np.diff(motif_beats)

    diffs = motif_intervals - anchor_intervals

    if np.allclose(diffs, 0, atol=tol):
        return {
            "preserved": False,
            "preserved_score": 0.0,
            "anchor_intervals": anchor_intervals.tolist(),
            "motif_intervals": motif_intervals.tolist(),
            "scaling_diffs": diffs.tolist(),
            "reason": "No change in rhythmic intervals"
        }

    median_diff = np.median(diffs)
    mad = np.mean(np.abs(diffs - median_diff))

    # Using both the range and the max absolute deviation as denominators.
    # This helps prevent instability when range is small, and avoids underestimating spread
    # when all values shift together. We take the maximum to ensure robustness.
    denom = np.max(diffs) - np.min(diffs) + tol
    alt_denom = np.max(np.abs(diffs)) + tol
    preserved_score = float(np.clip(1 - mad / max(denom, alt_denom), 0, 1))

    return {
        "preserved": preserved_score >= 0.75,
        "preserved_score": preserved_score,
        "anchor_intervals": anchor_intervals.tolist(),
        "motif_intervals": motif_intervals.tolist(),
        "scaling_diffs": diffs.tolist(),
        "reason": "Transformation detected via interval scaling homogeneity"
    }


In [ ]:
def evaluate_contour_inversion(anchor_df, motif_df, alignment, motion_eps=0.5):
    """
    Contour Inversion (CIT): melodic contour mirrored.
    - Operates on raw semitone intervals (no modulo).
    - Requires sign(Δp_m) == -sign(Δp_a) for all aligned steps.
    - Requires non-trivial motion (average magnitude > motion_eps) to avoid flat/noisy cases.

    Returns:
        dict with:
            - preserved : True/False/None
            - reason    : str
            - operations: list of {'anchor_interval', 'motif_interval'}
    """
    if len(alignment) < 2:
        return {"preserved": None, "operations": None, "reason": "Too few aligned notes"}

    anchor_pitches = anchor_df["pitch"].values
    motif_pitches  = motif_df["pitch"].values

    alignment = sorted(alignment, key=lambda x: x[0])
    anchor_idxs, motif_idxs = zip(*alignment)

    a = anchor_pitches[list(anchor_idxs)]
    m = motif_pitches[list(motif_idxs)]

    a_int = np.diff(a)
    m_int = np.diff(m)

    # sign flip check (note: -0 == 0, so zeros don't block inversion)
    a_sgn = np.sign(a_int)
    m_sgn = np.sign(m_int)
    sign_flip = np.array_equal(m_sgn, -a_sgn)

    # non-trivial motion threshold (no modulo)
    avg_motion = 0.5 * (np.mean(np.abs(a_int)) + np.mean(np.abs(m_int)))
    non_trivial_motion = avg_motion > motion_eps

    if sign_flip and non_trivial_motion:
        preserved = True
        reason = "Contour inverted with non-trivial motion"
    elif not sign_flip:
        preserved = False
        reason = "Contour direction not inverted"
    else:
        preserved = False
        reason = "Motion too small (near-flat intervals)"

    operations = [
        {"anchor_interval": float(ai), "motif_interval": float(mi)}
        for ai, mi in zip(a_int, m_int)
    ]

    return {
        "preserved": preserved,
        "operations": operations,
        "reason": reason
    }


In [ ]:
def evaluate_note_addition_removal(anchor_df, motif_df, alignment):
    """
    Evaluate true note addition/removal:
    - True if an unaligned note in motif/anchor is not equal in pitch to any of its neighbors.
    - Classifies location as start, middle, or end.

    Returns:
    - changed: True/False (was there a true addition or removal?)
    - type: "addition" / "removal" / None
    - location: "start" / "middle" / "end" / None
    """
    anchor_df = anchor_df.sort_values("onset").reset_index(drop=True)
    motif_df = motif_df.sort_values("onset").reset_index(drop=True)

    anchor_len = len(anchor_df)
    motif_len = len(motif_df)

    anchor_aligned = set(i for i, _ in alignment)
    motif_aligned = set(j for _, j in alignment)

    # Case 1: Addition — motif has an extra note
    if motif_len > anchor_len:
        added_idx = next(j for j in range(motif_len) if j not in motif_aligned)
        added_pitch = motif_df.loc[added_idx, "pitch"]

        neighbors = []
        if added_idx > 0:
            neighbors.append(motif_df.loc[added_idx - 1, "pitch"])
        if added_idx < motif_len - 1:
            neighbors.append(motif_df.loc[added_idx + 1, "pitch"])

        is_true_addition = all(added_pitch != n for n in neighbors)
        location = (
            "start" if added_idx == 0 else
            "end" if added_idx == motif_len - 1 else
            "middle"
        )

        return {
            "changed": is_true_addition,
            "type": "addition" if is_true_addition else None,
            "location": location if is_true_addition else None
        }

    # Case 2: Removal — anchor had a note that was dropped
    elif anchor_len > motif_len:
        removed_idx = next(i for i in range(anchor_len) if i not in anchor_aligned)
        removed_pitch = anchor_df.loc[removed_idx, "pitch"]

        neighbors = []
        if removed_idx > 0:
            neighbors.append(anchor_df.loc[removed_idx - 1, "pitch"])
        if removed_idx < anchor_len - 1:
            neighbors.append(anchor_df.loc[removed_idx + 1, "pitch"])

        is_true_removal = all(removed_pitch != n for n in neighbors)
        location = (
            "start" if removed_idx == 0 else
            "end" if removed_idx == anchor_len - 1 else
            "middle"
        )

        return {
            "changed": is_true_removal,
            "type": "removal" if is_true_removal else None,
            "location": location if is_true_removal else None
        }

    else:
        # No length difference = no addition or removal
        return {
            "changed": False,
            "type": None,
            "location": None
        }

In [ ]:
# role mapper (kept as-is)
def interval_type(semi):
    semi = abs(semi) % 12
    if semi in {1, 2}:  return "2"   # second
    if semi in {3, 4}:  return "3"   # third
    if semi == 5:       return "4"   # fourth
    if semi == 6:       return "TT"  # tritone
    if semi == 7:       return "5"   # fifth
    if semi in {8, 9}:  return "6"   # sixth
    if semi in {10,11}: return "7"   # seventh
    return "8"                       # octave/compound

# Canonicalize roles to enable quartal surrogacy (4th ≈ 5th)
def _canon_role(role):
    return "Q" if role in {"4", "5"} else role

def evaluate_interval_grammar(anchor_df, motif_df, alignment):
    """
    Interval-Grammar Transformation (IGT):
    - Map Δp to interval roles modulo 12 (maj/min collapsed; 4th<->5th surrogacy).
    - The multiset of roles is identical between anchor and motif.
    - The order of roles differs (i.e., not the same sequence).
    - Requires at least 2 intervals (>= 3 aligned notes).

    Returns:
        dict with:
            - preserved : True/False/None
            - reason    : str
            - anchor_roles, motif_roles : list[str]
            - anchor_intervals, motif_intervals : list[int]
            - alignment_indices : list[tuple]
    """
    if len(alignment) < 2:
        return {
            "preserved": None,
            "anchor_intervals": [],
            "motif_intervals": [],
            "anchor_roles": [],
            "motif_roles": [],
            "alignment_indices": alignment,
            "reason": "Too few aligned notes"
        }

    alignment = sorted(alignment, key=lambda x: x[0])
    anchor_idxs, motif_idxs = zip(*alignment)

    a_p = anchor_df.loc[list(anchor_idxs), "pitch"].values
    m_p = motif_df.loc[list(motif_idxs), "pitch"].values

    a_int = np.diff(a_p)              # raw semitone diffs (sign irrelevant after role)
    m_int = np.diff(m_p)

    a_roles = [_canon_role(interval_type(s)) for s in a_int]
    m_roles = [_canon_role(interval_type(s)) for s in m_int]

    # Multiset equality (grammar preserved)
    same_multiset = Counter(a_roles) == Counter(m_roles)
    # Sequence order must differ (avoid overlap with Intervalic-Preserving)
    different_order = (a_roles != m_roles)

    preserved = bool(same_multiset and different_order)
    reason = None
    if not preserved:
        if not same_multiset:
            reason = "Interval-grammar differs (multiset mismatch)"
        elif not different_order:
            reason = "Interval-grammar order unchanged (identical sequence)"

    return {
        "preserved": preserved,
        "anchor_intervals": [int(x) for x in a_int],
        "motif_intervals":  [int(x) for x in m_int],
        "anchor_roles": a_roles,
        "motif_roles": m_roles,
        "alignment_indices": alignment,
        "reason": reason
    }


In [ ]:
def evaluate_harmony_preserving(anchor_df, motif_df, alignment):
    """
    Evaluate Harmony-Preserving Transformation between aligned motifs using encoded harmony features.

    Returns:
    - preserved: True if functional zones match
    - anchor_zones / motif_zones: collapsed functional traces
    - transformations: list of symbolic harmonic changes
    - alignment_indices: list of (anchor_idx, motif_idx) used
    """
    if len(alignment) < 1:
        return {
            "preserved": None,
            "anchor_zones": [],
            "motif_zones": [],
            "transformations": [],
            "alignment_indices": []
        }

    # Correct functional zone assignment using both scale degree and flags
    def map_functional_zone(row):
        sd = int(row['scale_degree_semitone_offset'])
        if sd in [0, 4, 8]:  # I, vi, iii
            return 'T'
        elif sd in [2, 5]:   # ii, IV
            return 'PD'
        elif sd in [7, 10, 11]:  # V, vii°, etc.
            return 'D'
        if row['is_neapolitan'] == 1:
            return 'PD'
        if row['is_augmented_sixth'] == 1:
            return 'D'
        return 'X'

    # Sort alignment
    alignment = sorted(alignment, key=lambda x: x[0])
    anchor_idxs, motif_idxs = zip(*alignment)
    alignment_indices = list(zip(anchor_idxs, motif_idxs))

    # Extract aligned harmony info
    encoded_cols = [
        'scale_degree_semitone_offset', 'is_major', 'is_minor',
        'is_diminished', 'is_augmented', 'is_seventh',
        'chord_inversion', 'is_augmented_sixth', 'is_neapolitan'
    ]

    anchor_harmony = anchor_df.loc[list(anchor_idxs), encoded_cols].reset_index(drop=True)
    motif_harmony = motif_df.loc[list(motif_idxs), encoded_cols].reset_index(drop=True)

    # Functional zone traces (collapsed)
    anchor_zones = []
    for _, row in anchor_harmony.iterrows():
        zone = map_functional_zone(row)
        if not anchor_zones or anchor_zones[-1] != zone:
            anchor_zones.append(zone)

    motif_zones = []
    for _, row in motif_harmony.iterrows():
        zone = map_functional_zone(row)
        if not motif_zones or motif_zones[-1] != zone:
            motif_zones.append(zone)



    # Symbolic transformations
    transformations = []

    for i in range(len(anchor_harmony)):
        a_row = anchor_harmony.iloc[i]
        m_row = motif_harmony.iloc[i]

        # Modal shift
        if a_row['is_major'] != m_row['is_major'] or a_row['is_minor'] != m_row['is_minor']:
            transformations.append("modal_shift")

        # Quality change
        if (a_row['is_diminished'], a_row['is_augmented']) != (m_row['is_diminished'], m_row['is_augmented']):
            transformations.append("change_quality")

        # Seventh added/removed
        if a_row['is_seventh'] != m_row['is_seventh']:
            action = 'add' if m_row['is_seventh'] == 1 else 'remove'
            transformations.append(f"{action}_seventh")

        # Inversion change
        if a_row['chord_inversion'] != m_row['chord_inversion']:
            transformations.append("change_inversion")

        # Chromatic function added/removed
        for tag in ['is_augmented_sixth', 'is_neapolitan']:
            if a_row[tag] != m_row[tag]:
                action = 'add' if m_row[tag] == 1 else 'remove'
                transformations.append(f"{action}_{tag[3:]}")
        
        preserved = anchor_zones == motif_zones and len(transformations) > 0

    return {
        "preserved": preserved,
        "anchor_zones": anchor_zones,
        "motif_zones": motif_zones,
        "transformations": sorted(set(transformations)),
        "alignment_indices": alignment_indices
    }


In [ ]:
# Map semitone distances to interval roles (ignoring exact size)
INTERVAL_ROLE_DICT = {
    0: 'root',
    1: 'second', 2: 'second',
    3: 'third', 4: 'third',
    5: 'fourth',
    6: 'tritone',
    7: 'fifth',
    8: 'sixth', 9: 'sixth',
    10: 'seventh', 11: 'seventh'
}

def evaluate_intervalic_preserving(anchor_df, motif_df, alignment, tol=1e-5):
    """
    Check if the interval *roles* between aligned notes are preserved (mod 12).
    Returns False if all pitch classes (mod 12) are equal across alignment.
    """
    if len(alignment) < 2:
        return {
            "preserved": None,
            "anchor_intervals": [],
            "motif_intervals": [],
            "anchor_roles": [],
            "motif_roles": [],
            "anchor_pitches": [],
            "motif_pitches": [],
            "alignment_indices": alignment,
            "reason": "Too few aligned notes"
        }

    alignment = sorted(alignment, key=lambda x: x[0])
    anchor_idxs, motif_idxs = zip(*alignment)

    anchor_pitches = anchor_df.loc[list(anchor_idxs), "pitch"].values
    motif_pitches = motif_df.loc[list(motif_idxs), "pitch"].values
    
    # Check if all pitch classes are equal
    if np.allclose(anchor_pitches, motif_pitches, atol=tol):
        return {
            "preserved": False,
            "anchor_intervals": [],
            "motif_intervals": [],
            "anchor_roles": [],
            "motif_roles": [],
            "anchor_pitches": anchor_pitches.tolist(),
            "motif_pitches": motif_pitches.tolist(),
            "alignment_indices": alignment,
            "reason": "All pitch classes are equal – no transformation"
        }

    # Compute intervals mod 12
    anchor_intervals = np.diff(anchor_pitches) % 12
    motif_intervals = np.diff(motif_pitches) % 12

    # Map to interval roles and raise error if unknown
    try:
        anchor_roles = [INTERVAL_ROLE_DICT[int(x)] for x in anchor_intervals]
        motif_roles = [INTERVAL_ROLE_DICT[int(x)] for x in motif_intervals]
    except KeyError as e:
        raise ValueError(f"Unknown interval encountered: {e}")

    preserved = anchor_roles == motif_roles

    return {
        "preserved": preserved,
        "anchor_intervals": anchor_intervals.tolist(),
        "motif_intervals": motif_intervals.tolist(),
        "anchor_roles": anchor_roles,
        "motif_roles": motif_roles,
        "anchor_pitches": anchor_pitches.tolist(),
        "motif_pitches": motif_pitches.tolist(),
        "alignment_indices": alignment,
        "reason": None if preserved else "Interval roles differ"
    }


In [ ]:
TRANSFORMATION_FAMILIES = [
    "identity",
    "contour",
    "salient_leap",
    "rhythm",
    "contour_inversion",
    "interval_grammer",
    "note_edit",
    "harmony",
    "intervalic"
]
FAMILY_TO_INDEX = {name: i for i, name in enumerate(TRANSFORMATION_FAMILIES)}


In [ ]:
def compute_sentence_label_matrix(final_segment_motif_dict, segment_anchor_motifs):
    sentence_label_matrix = []
    motif_metadata = []

    # Presort by segment_id to ensure consistent ordering
    for segment_id in sorted(final_segment_motif_dict):
        class_dict = final_segment_motif_dict[segment_id]
        all_motifs = []

        # Step 1: Flatten all motif instances across classes
        for motif_class, motif_instances in class_dict.items():
            if motif_class not in segment_anchor_motifs.get(segment_id, {}):
                continue
            anchor_df = segment_anchor_motifs[segment_id][motif_class]
            for motif_df in motif_instances:
                onset = motif_df["onset"].min()
                all_motifs.append((onset, segment_id, motif_class, anchor_df, motif_df))

        # Step 2: Sort motifs by onset within the segment
        all_motifs.sort(key=lambda x: x[0])

        # Step 3: Create label vectors in onset order
        for onset, segment_id, motif_class, anchor_df, motif_df in all_motifs:
            label_vector = np.zeros(len(TRANSFORMATION_FAMILIES), dtype=int)
            alignment = align_motifs(anchor_df, motif_df)

            if evaluate_identity_transformation(anchor_df, motif_df, alignment)["preserved"]:
                label_vector[FAMILY_TO_INDEX["identity"]] = 1
            if evaluate_contour_preserving(anchor_df, motif_df, alignment)["preserved"]:
                label_vector[FAMILY_TO_INDEX["contour"]] = 1
            if evaluate_salient_leap_preserving(anchor_df, motif_df, alignment)["preserved"]:
                label_vector[FAMILY_TO_INDEX["salient_leap"]] = 1
            if evaluate_rhythm_preserving(anchor_df, motif_df, alignment)["preserved"]:
                label_vector[FAMILY_TO_INDEX["rhythm"]] = 1
            if evaluate_contour_inversion(anchor_df, motif_df, alignment)["preserved"]:
                label_vector[FAMILY_TO_INDEX["contour_inversion"]] = 1
            if evaluate_interval_grammar(anchor_df, motif_df, alignment)["preserved"]:
                label_vector[FAMILY_TO_INDEX["interval_grammer"]] = 1
            if evaluate_note_addition_removal(anchor_df, motif_df, alignment)["changed"]:
                label_vector[FAMILY_TO_INDEX["note_edit"]] = 1
            if evaluate_harmony_preserving(anchor_df, motif_df, alignment)["preserved"]:
                label_vector[FAMILY_TO_INDEX["harmony"]] = 1
            if evaluate_intervalic_preserving(anchor_df, motif_df, alignment)["preserved"]:
                label_vector[FAMILY_TO_INDEX["intervalic"]] = 1

            sentence_label_matrix.append(label_vector)
            motif_metadata.append({
                "segment_id": segment_id,
                "motif_class": motif_class,
                "onset": onset
            })

    return np.array(sentence_label_matrix), pd.DataFrame(motif_metadata).sort_values(["segment_id", "onset"]).reset_index(drop=True)
        


In [ ]:
def save_transformation_outputs(
    sonata_id,
    motif_features,
    sentence_label_matrix,
    motif_metadata,
    base_path=BASE_DIR
):
    """
    Saves transformation outputs (motif_features, sentence_label_matrix, motif_metadata)
    to disk in structured subfolders.

    Parameters:
    - sonata_id: string like '20'
    - motif_features: DataFrame
    - sentence_label_matrix: NumPy array
    - motif_metadata: DataFrame
    - base_path: root base path for saving
    """
    # Root directory for all outputs
    root_save_dir = base_path / "processed" / "transformations_with_motifs"
    root_save_dir.mkdir(parents=True, exist_ok=True)

    # Items to save and their variable names
    save_items = {
        "motif_features": motif_features,
        "sentence_label_matrix": sentence_label_matrix,
        "motif_metadata": motif_metadata
    }

    # Iterate and save each item
    for var_name, data in save_items.items():
        subfolder = root_save_dir / var_name
        subfolder.mkdir(parents=True, exist_ok=True)

        filename = f"sonata{sonata_id}_{var_name}"
        file_path = subfolder / filename

        if isinstance(data, dict):
            with open(file_path.with_suffix(".pkl"), "wb") as f:
                pickle.dump(data, f)
        elif isinstance(data, pd.DataFrame):
            data.to_feather(file_path.with_suffix(".feather"))
        elif isinstance(data, np.ndarray):
            np.save(file_path.with_suffix(".npy"), data)
        else:
            raise TypeError(f"Unsupported type for variable '{var_name}': {type(data)}")

In [ ]:
import pandas as pd

SONATA_RANGE = range(1, 33)

for i in SONATA_RANGE:
    sonata_num = f"{i:02d}"

    # === Check if already processed ===
    root_save_dir = BASE_DIR / "processed" / "transformations_with_motifs"
    required_outputs = [
        root_save_dir / "motif_features" / f"sonata{sonata_num}_motif_features.feather",
        root_save_dir / "sentence_label_matrix" / f"sonata{sonata_num}_sentence_label_matrix.npy",
        root_save_dir / "motif_metadata" / f"sonata{sonata_num}_motif_metadata.feather"
    ]

    # Skip sonata if all outputs already exist
    if all(path.exists() for path in required_outputs):
        print(f"Skipping sonata {sonata_num} (already processed).")
        continue

    print(f"Processing sonata {sonata_num}...")

    # === Load preprocessed feather ===
    feather_path = PREPROCESSED_DIR / f"extracted_{sonata_num}-1.feather"
    df = pd.read_feather(feather_path)

    # Force numeric values in onset and offset columns
    df["onset"] = pd.to_numeric(df["onset"], errors="coerce")
    df["offset"] = pd.to_numeric(df["offset"], errors="coerce")

    # === Filter only motif notes ===
    selected_features = [
        # General type
        "type",

        # Motif type
        "motif",

        # Identification & Timing
        "pitch", "pitchname", "relative_pitch_scale", "octave", "clef", "onset", "offset",
        "start_meas", "end_meas", "scaled_duration", "beat", "scaled_beat",
        "timesig_numerator", "timesig_denominator", "timesig_ratio", "normalized_ioi",

        # Harmonic Context
        "scale_degree_semitone_offset", "secondary_scale_degree_semitone_offset",
        "is_major", "is_minor", "is_diminished", "is_augmented", "is_seventh",
        "is_neapolitan", "is_augmented_sixth", "chord_inversion", "harmonic_complexity",

        # Dynamics & Expressive
        "ordinal_dynamic", "is_accent", "is_tenuto", "is_staccato",
        "is_ornament", "grace", "is_cresc", "is_dim", "is_slur",

        # Key Context
        "localkey_fifths", "localkey_mode"
    ]

    df = df[selected_features]
    df_motifs_only = df[df["motif"].notna()].copy()
    df_motifs_only.reset_index(drop=True, inplace=True)

    # === Match motif instances ===
    motif_folder = MOTIF_DIR / sonata_num
    motif_groups = match_motif_instances(df_motifs_only, motif_folder)

    # === Sort instances by onset ===
    motif_groups = sort_motif_instances(motif_groups)

    # === Sentence segmentation ===
    sentence_boundaries = detect_sentence_boundaries(df)
    segment_dfs = build_segment_dfs(df, sentence_boundaries)

    # === Assign motifs to segments ===
    final_segment_motif_dict = assign_motifs_to_segments(motif_groups, sentence_boundaries)

    # === Compute motif-level features ===
    motif_features = compute_motif_level_features(final_segment_motif_dict)

    # === Compute anchor motifs ===
    segment_anchor_motifs, anchor_metadata = compute_anchor_motifs(final_segment_motif_dict)

    # === Compute sentence-level label matrix ===
    sentence_label_matrix, motif_metadata = compute_sentence_label_matrix(
        final_segment_motif_dict,
        segment_anchor_motifs
    )

    # === Save outputs ===
    save_transformation_outputs(
        sonata_id=sonata_num,
        motif_features=motif_features,
        sentence_label_matrix=sentence_label_matrix,
        motif_metadata=motif_metadata
    )